In [5]:
import os
from os import path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pickle

import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.metrics import mean_squared_error
from scipy import stats

In [6]:
dataset_id = 202
data_folder = path.join('./data', str(dataset_id))
output_folder = path.join('./output', str(dataset_id))
if not path.exists(output_folder):
    os.makedirs(output_folder)

In [7]:
with open(path.join(data_folder, 'users.pickle'), 'rb') as f:
    users = pickle.load(f)

In [8]:
S1_BELIEFS = 'S1_Q1 + S1_Q2 + S1_Q3 + S1_Q4 + gender'
S2_BELIEFS = 'S2_Q1 + S2_Q2 + S2_Q3 + S2_Q4 + gender'
S1_NEEDED = ['S1_Q1', 'S1_Q2', 'S1_Q3', 'S1_Q4', 'gender']
S2_NEEDED = ['S2_Q1', 'S2_Q2', 'S2_Q3', 'S2_Q4', 'gender']

belief_cols = ['S1_Q1', 'S1_Q2', 'S1_Q3', 'S1_Q4', 'S2_Q1', 'S2_Q2', 'S2_Q3', 'S2_Q4']

# belief_refs = users[belief_cols].mean() # Grounding at the belief means (all four raw beliefs = mean)
belief_refs = pd.Series(1, index=belief_cols) # Grounding at the belief floor (all four raw beliefs = 1)

qb_se_scale = 'X2'
#qb_se_scale = None

users['ever_quarantine'] = (users['quarantine'] > 0).astype(int)

def fit_gee(formula, needed=None):
    d = daily.dropna(subset=needed) if needed else daily
    m = sm.GEE.from_formula(formula, groups='id', data=d,
                            family=sm.families.Binomial(),
                            cov_struct=sm.cov_struct.Exchangeable()).fit()
    print(f'N obs={int(m.nobs)}  participants={d["id"].nunique()}  '
          f'within-participant alpha={m.cov_struct.dep_params:.3f}')
    return m

def fit_quasibinomial(formula, needed=None, se_scale='X2'):
    required = ['quarantine_rate', 'total_trials', 'group']
    if needed:
        required += needed
    required = list(dict.fromkeys(required))
    data = aggregate.dropna(subset=required).copy()

    m = smf.glm(
        formula=formula,
        data=data,
        family=sm.families.Binomial(),
        var_weights=data['total_trials'],
    ).fit(scale=se_scale)

    print(
        f'N participants={int(m.nobs)}  '
        f'decisions={int(data["total_trials"].sum())}  '
        f'Pearson scale={m.scale:.3f}'
    )

    return m

def skeptic_contrast(model, survey, gender=None, gender_weight=None):
    """
    Barrier effect at the belief floor (all four raw beliefs = 1).
    gender=0: men
    gender=1: women
    gender=None: standardized over the sample gender distribution
    """

    terms = [f'{1 - belief_refs[f"{survey}_Q{i}"]:+.6f}*{survey}_Q{i}:C(group)[T.2]' for i in range(1, 5)]
    expression = 'C(group)[T.2] ' + ' '.join(terms)

    if gender is None:
        if gender_weight is None:
            needed = S1_NEEDED if survey == 'S1' else S2_NEEDED
            analysis_users = users.dropna(subset=needed)
            gender_weight = analysis_users['gender'].mean()

        expression += (f' + {gender_weight:.8f}*gender:C(group)[T.2]')
    elif gender == 1:
        expression += ' + gender:C(group)[T.2]'
    elif gender != 0:
        raise ValueError('gender must be 0, 1, or None')

    test = model.t_test(expression + ' = 0')

    return {
        'coef': float(np.asarray(test.effect).squeeze()),
        'se': float(np.asarray(test.sd).squeeze()),
        'p_value': float(np.asarray(test.pvalue).squeeze()),
        'gender_weight': gender_weight if gender is None else gender,
    }

## GEE models for daily quarantine decisions

Daily decisions are repeated measures within participants. These GEE (generalized estimating equations) models fit the same mean structure as the pre-registered specification with exchangeable within-participant correlation and robust standard errors.

https://www.publichealth.columbia.edu/research/population-health-methods/repeated-measures-analysis

Belief items retain their original Likert coding. Therefore, `C(group)` main effects in models with belief interactions are evaluated at the Likert floor (`belief_refs = 1`), representing the pre-registered "skeptic" contrast. Joint tests of the interaction terms are reported separately below.

In [23]:
n_days = len([c for c in users.columns if c.startswith('quarantine_day')])
carry = ['id', 'group', 'gender',
         'S1_Q1', 'S1_Q2', 'S1_Q3', 'S1_Q4', 'S2_Q1', 'S2_Q2', 'S2_Q3', 'S2_Q4']

frames = []
for i in range(n_days):
    s = users[carry].copy()
    s['day'] = i
    s['q'] = users[f'quarantine_day{i}'].values
    s['nq'] = users[f'no_quarantine_day{i}'].values
    frames.append(s)

daily = pd.concat(frames, ignore_index=True)
daily = daily[(daily['q'] + daily['nq']) > 0].drop(columns='nq').reset_index(drop=True)

for c in belief_cols:
    daily[c] = daily[c] - belief_refs[c]

print(f'{len(daily)} participant-days from {daily["id"].nunique()} participants')
print(f'ever quarantined: {users["ever_quarantine"].mean():.1%}')

1611 participant-days from 403 participants
ever quarantined: 26.3%


In [24]:
print("--- Barrier main effect (full behavioral cohort) ---")
gee_h1 = fit_gee('q ~ C(group)')
print(gee_h1.summary().tables[1])

--- Barrier main effect (full behavioral cohort) ---
N obs=1611  participants=403  within-participant alpha=0.164
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept        -2.0721      0.166    -12.460      0.000      -2.398      -1.746
C(group)[T.2]    -0.1016      0.224     -0.454      0.650      -0.540       0.337


In [11]:
print("--- H2: S1 beliefs x group ---")
gee_h2 = fit_gee(f'q ~ ({S1_BELIEFS}) * C(group)', S1_NEEDED)
print(gee_h2.summary().tables[1])

wald_h2 = gee_h2.wald_test(
    'S1_Q1:C(group)[T.2] = S1_Q2:C(group)[T.2] = '
    'S1_Q3:C(group)[T.2] = S1_Q4:C(group)[T.2] = 0', scalar=True)
print(f'H2 joint Wald: chi2={wald_h2.statistic:.3f}  df=4  p={wald_h2.pvalue:.4f}')

all_skeptics_h1 = skeptic_contrast(gee_h2, 'S1')
print(
    'Barrier effect for all S1 skeptics, '
    f'standardized to {all_skeptics_h1["gender_weight"]:.1%} women: '
    f'coef={all_skeptics_h1["coef"]:+.4f} '
    f'SE={all_skeptics_h1["se"]:.4f} '
    f'p={all_skeptics_h1["p_value"]:.4f} '
    f'OR={np.exp(all_skeptics_h1["coef"]):.3f}'
)

--- H2: S1 beliefs x group ---
N obs=1449  participants=327  within-participant alpha=0.192
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept               -2.0894      0.887     -2.356      0.018      -3.828      -0.351
C(group)[T.2]           -0.3469      1.054     -0.329      0.742      -2.412       1.718
S1_Q1                    0.0215      0.143      0.151      0.880      -0.258       0.301
S1_Q1:C(group)[T.2]      0.0946      0.191      0.496      0.620      -0.279       0.468
S1_Q2                    0.1986      0.137      1.446      0.148      -0.071       0.468
S1_Q2:C(group)[T.2]     -0.1420      0.185     -0.769      0.442      -0.504       0.220
S1_Q3                   -0.2160      0.161     -1.345      0.179      -0.531       0.099
S1_Q3:C(group)[T.2]      0.1667      0.222      0.749      0.454      -0.269       0.603
S1_Q4             

In [12]:
print("--- H2b: S2 beliefs x group ---")
gee_h2b = fit_gee(f'q ~ ({S2_BELIEFS}) * C(group)', S2_NEEDED)
print(gee_h2b.summary().tables[1])

wald_h2b = gee_h2b.wald_test(
    'S2_Q1:C(group)[T.2] = S2_Q2:C(group)[T.2] = '
    'S2_Q3:C(group)[T.2] = S2_Q4:C(group)[T.2] = 0', scalar=True)
print(f'H2b joint Wald: chi2={wald_h2b.statistic:.3f}  df=4  p={wald_h2b.pvalue:.4f}')

all_skeptics_h2 = skeptic_contrast(gee_h2b, 'S2')
print(
    'Barrier effect for all S2 skeptics, '
    f'standardized to {all_skeptics_h2["gender_weight"]:.1%} women: '
    f'coef={all_skeptics_h2["coef"]:+.4f} '
    f'SE={all_skeptics_h2["se"]:.4f} '
    f'p={all_skeptics_h2["p_value"]:.4f} '
    f'OR={np.exp(all_skeptics_h2["coef"]):.3f}'
)

--- H2b: S2 beliefs x group ---
N obs=1279  participants=250  within-participant alpha=0.210
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept               -2.5391      0.718     -3.537      0.000      -3.946      -1.132
C(group)[T.2]           -2.8305      1.305     -2.169      0.030      -5.388      -0.273
S2_Q1                    0.1019      0.203      0.502      0.616      -0.296       0.500
S2_Q1:C(group)[T.2]      0.3458      0.316      1.093      0.275      -0.275       0.966
S2_Q2                   -0.0275      0.137     -0.200      0.841      -0.297       0.242
S2_Q2:C(group)[T.2]     -0.0332      0.259     -0.128      0.898      -0.541       0.475
S2_Q3                    0.0983      0.144      0.685      0.493      -0.183       0.380
S2_Q3:C(group)[T.2]      0.0448      0.211      0.212      0.832      -0.369       0.458
S2_Q4            

## Health Belief Model (GEE)

Additive structure as pre-registered, with the barrier entering as one of the HBM constructs.
The interaction variant is a test of H2/H2b, not an alternative HBM: Step 2 of the protocol
applies a single set of coefficients to the S1 real-life beliefs, which requires one set of
betas. The null H2/H2b results license pooling across arms.

In [13]:
print("--- HBM from in-game beliefs (S2) ---")
gee_hbm2 = fit_gee(f'q ~ {S2_BELIEFS} + C(group)', S2_NEEDED)
print(gee_hbm2.summary().tables[1])

--- HBM from in-game beliefs (S2) ---
N obs=1279  participants=250  within-participant alpha=0.185
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept        -3.3973      0.664     -5.120      0.000      -4.698      -2.097
C(group)[T.2]    -0.2816      0.292     -0.964      0.335      -0.854       0.291
S2_Q1             0.2240      0.152      1.478      0.139      -0.073       0.521
S2_Q2            -0.0276      0.122     -0.227      0.820      -0.266       0.211
S2_Q3             0.1018      0.101      1.008      0.313      -0.096       0.300
S2_Q4             0.0476      0.125      0.380      0.704      -0.198       0.293
gender            0.4915      0.354      1.387      0.165      -0.203       1.186


In [14]:
print("--- HBM from real-life beliefs (S1) ---")
gee_hbm1 = fit_gee(f'q ~ {S1_BELIEFS} + C(group)', S1_NEEDED)
print(gee_hbm1.summary().tables[1])

--- HBM from real-life beliefs (S1) ---
N obs=1449  participants=327  within-participant alpha=0.194
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept        -2.2566      0.506     -4.463      0.000      -3.248      -1.266
C(group)[T.2]    -0.0721      0.252     -0.287      0.774      -0.565       0.421
S1_Q1             0.0697      0.093      0.746      0.456      -0.114       0.253
S1_Q2             0.1327      0.091      1.453      0.146      -0.046       0.312
S1_Q3            -0.1230      0.111     -1.109      0.268      -0.340       0.094
S1_Q4            -0.0604      0.128     -0.473      0.636      -0.311       0.190
gender            0.5401      0.271      1.992      0.046       0.009       1.072


## Adoption: did the participant ever quarantine?

45% of participants never quarantined, so the rate models are fitted to a heavily zero-inflated
outcome. Modelling adoption separately asks whether the barrier changed *whether* people
quarantined at all, as distinct from how often. One row per participant, so no clustering
correction is needed.

In [15]:
for label, formula, cols in [
    ('barrier',            'ever_quarantine ~ C(group)',          ['ever_quarantine', 'group']),
    ('gender',             'ever_quarantine ~ gender',            ['ever_quarantine', 'gender']),
    ('barrier + gender',   'ever_quarantine ~ gender + C(group)', ['ever_quarantine', 'gender', 'group']),
    ('barrier x gender',   'ever_quarantine ~ gender * C(group)', ['ever_quarantine', 'gender', 'group']),
]:
    d = users[cols].dropna()
    r = smf.glm(formula, data=d, family=sm.families.Binomial()).fit()
    print(f'--- {label} (N={int(r.nobs)}) ---')
    for t in [x for x in r.params.index if x != 'Intercept']:
        print(f'    {t:24s} OR={np.exp(r.params[t]):.2f}  p={r.pvalues[t]:.4f}')


--- barrier (N=403) ---
    C(group)[T.2]            OR=1.13  p=0.5787
--- gender (N=327) ---
    gender                   OR=2.72  p=0.0002
--- barrier + gender (N=327) ---
    C(group)[T.2]            OR=1.24  p=0.4094
    gender                   OR=2.76  p=0.0002
--- barrier x gender (N=327) ---
    C(group)[T.2]            OR=1.10  p=0.8227
    gender                   OR=2.50  p=0.0231
    gender:C(group)[T.2]     OR=1.20  p=0.7369


## Quasi-binomial models for participant quarantine propensity

These participant-level models parallel the daily-decision GEE analyses. The outcome is each participant's overall quarantine rate.

Models use:

* `total_trials` as precision weights;
* Pearson scaling (`scale='X2'`) for extra-binomial variation;
* the same complete-case samples as the corresponding GEE belief models; and
* belief items shifted by 1 so zero represents the original Likert floor.

In interaction models, the group coefficient represents the barrier effect at the belief floor for men (`gender = 0`). Explicit contrasts standardize this floor-profile effect over the gender distribution of the relevant analysis sample.

In [26]:
aggregate = users.copy()
for col in belief_cols:
   aggregate[col] = aggregate[col] - 1

In [25]:
print('--- Quasi-binomial H1: full behavioral cohort ---')
qb_h1 = fit_quasibinomial('quarantine_rate ~ C(group)', se_scale=qb_se_scale)
print(qb_h1.summary().tables[1])

--- Quasi-binomial H1: full behavioral cohort ---
N participants=403  decisions=1615  Pearson scale=1.946
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept        -2.2461      0.174    -12.902      0.000      -2.587      -1.905
C(group)[T.2]    -0.0755      0.240     -0.314      0.753      -0.546       0.395


In [18]:
print('--- Quasi-binomial H2: S1 beliefs x group ---')
qb_h2 = fit_quasibinomial(f'quarantine_rate ~ ({S1_BELIEFS}) * C(group)', S1_NEEDED, se_scale=qb_se_scale)
print(qb_h2.summary().tables[1])

qb_wald_h2 = qb_h2.wald_test(
    'S1_Q1:C(group)[T.2] = S1_Q2:C(group)[T.2] = '
    'S1_Q3:C(group)[T.2] = S1_Q4:C(group)[T.2] = 0', scalar=True)
print(f'H2 joint Wald: chi2={qb_wald_h2.statistic:.3f}  df=4  p={qb_wald_h2.pvalue:.4f}')

all_skeptics_s1 = skeptic_contrast(qb_h2, 'S1')
print(
    'Barrier effect for all S1 skeptics, '
    f'standardized to {all_skeptics_s1["gender_weight"]:.1%} women: '
    f'coef={all_skeptics_s1["coef"]:+.4f} '
    f'SE={all_skeptics_s1["se"]:.4f} '
    f'p={all_skeptics_s1["p_value"]:.4f} '
    f'OR={np.exp(all_skeptics_s1["coef"]):.3f}'
)

--- Quasi-binomial H2: S1 beliefs x group ---
N participants=327  decisions=1451  Pearson scale=2.132
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept               -2.4229      0.894     -2.709      0.007      -4.176      -0.670
C(group)[T.2]           -0.3014      1.184     -0.255      0.799      -2.622       2.019
S1_Q1                    0.0761      0.154      0.493      0.622      -0.227       0.379
S1_Q1:C(group)[T.2]      0.0548      0.216      0.254      0.799      -0.368       0.478
S1_Q2                    0.1353      0.147      0.923      0.356      -0.152       0.423
S1_Q2:C(group)[T.2]     -0.0350      0.217     -0.162      0.872      -0.460       0.390
S1_Q3                   -0.1680      0.166     -1.012      0.312      -0.494       0.158
S1_Q3:C(group)[T.2]      0.1254      0.216      0.582      0.561      -0.297       0.548
S1_Q4   

In [19]:
print('--- Quasi-binomial H2b: S2 beliefs x group ---')
qb_h2b = fit_quasibinomial(f'quarantine_rate ~ ({S2_BELIEFS}) * C(group)', S2_NEEDED, se_scale=qb_se_scale)
print(qb_h2b.summary().tables[1])

qb_wald_h2b = qb_h2b.wald_test(
    'S2_Q1:C(group)[T.2] = S2_Q2:C(group)[T.2] = '
    'S2_Q3:C(group)[T.2] = S2_Q4:C(group)[T.2] = 0', scalar=True)
print(f'H2b joint Wald: chi2={qb_wald_h2b.statistic:.3f}  df=4  p={qb_wald_h2b.pvalue:.4f}')

all_skeptics_h2b = skeptic_contrast(qb_h2b, 'S2')
print(
    'Barrier effect for all S2 skeptics, '
    f'standardized to {all_skeptics_h2b["gender_weight"]:.1%} women: '
    f'coef={all_skeptics_h2b["coef"]:+.4f} '
    f'SE={all_skeptics_h2b["se"]:.4f} '
    f'p={all_skeptics_h2b["p_value"]:.4f} '
    f'OR={np.exp(all_skeptics_h2b["coef"]):.3f}'
)

--- Quasi-binomial H2b: S2 beliefs x group ---
N participants=250  decisions=1280  Pearson scale=2.490
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept               -2.8630      0.859     -3.331      0.001      -4.547      -1.179
C(group)[T.2]           -2.8567      1.482     -1.927      0.054      -5.761       0.048
S2_Q1                    0.1177      0.223      0.528      0.598      -0.319       0.555
S2_Q1:C(group)[T.2]      0.2315      0.313      0.741      0.459      -0.381       0.844
S2_Q2                   -0.0139      0.169     -0.083      0.934      -0.345       0.317
S2_Q2:C(group)[T.2]      0.0268      0.246      0.109      0.913      -0.454       0.508
S2_Q3                    0.1109      0.184      0.602      0.547      -0.250       0.472
S2_Q3:C(group)[T.2]      0.0995      0.276      0.361      0.718      -0.441       0.640
S2_Q4  

## Health Belief Model: quasi-binomial specification

These additive models parallel the GEE Health Belief Models. Group assignment represents the experimentally manipulated barrier, while the four belief items and gender enter as participant-level predictors.

In [20]:
print("--- HBM from in-game beliefs (S2) ---")
qb_hbm2 = fit_quasibinomial(f'quarantine_rate ~ {S2_BELIEFS} + C(group)', S2_NEEDED, se_scale=qb_se_scale)
print(qb_hbm2.summary().tables[1])

--- HBM from in-game beliefs (S2) ---
N participants=250  decisions=1280  Pearson scale=2.284
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept        -3.8064      0.709     -5.370      0.000      -5.196      -2.417
C(group)[T.2]    -0.2011      0.321     -0.627      0.531      -0.830       0.428
S2_Q1             0.1901      0.147      1.297      0.195      -0.097       0.477
S2_Q2             0.0145      0.115      0.126      0.900      -0.212       0.241
S2_Q3             0.1424      0.130      1.096      0.273      -0.112       0.397
S2_Q4             0.0827      0.145      0.571      0.568      -0.201       0.366
gender            0.3770      0.348      1.082      0.279      -0.306       1.060


In [21]:
print("--- HBM from real-life beliefs (S1) ---")
qb_hbm1 = fit_quasibinomial(f'quarantine_rate ~ {S1_BELIEFS} + C(group)', S1_NEEDED, se_scale=qb_se_scale)
print(qb_hbm1.summary().tables[1])

--- HBM from real-life beliefs (S1) ---
N participants=327  decisions=1451  Pearson scale=2.124
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept        -2.5750      0.606     -4.247      0.000      -3.763      -1.387
C(group)[T.2]    -0.0367      0.278     -0.132      0.895      -0.581       0.508
S1_Q1             0.1032      0.106      0.971      0.332      -0.105       0.312
S1_Q2             0.1157      0.107      1.081      0.280      -0.094       0.326
S1_Q3            -0.0950      0.105     -0.902      0.367      -0.301       0.111
S1_Q4            -0.0399      0.142     -0.281      0.778      -0.318       0.238
gender            0.4259      0.293      1.452      0.147      -0.149       1.001


## Exploratory S3 associations with quarantine behavior

S3 was administered after gameplay, so these models assess retrospective behavioral correspondence rather than prospective prediction or causation. For each of S3-Q1--Q4, we report a participant-level Spearman correlation, a group-adjusted participant-day GEE association, and a Pearson-scaled participant-level quasi-binomial association. Separate interaction models estimate the item slope in each randomized group and formally test item-by-group heterogeneity. Holm correction is applied separately across the four items for each main-association and interaction family.

In [22]:
from pathlib import Path
import pickle
import re

import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

s3_items = {
    'S3_Q1': 'Perceived realism',
    'S3_Q2': 'Concern about virtual infection',
    'S3_Q3': 'Real-life correspondence of quarantine decisions',
    'S3_Q4': 'Difficulty understanding game information',
}

with Path('data/202/users.pickle').open('rb') as handle:
    s3_users = pickle.load(handle).copy()

day_numbers = sorted(
    int(match.group(1))
    for column in s3_users.columns
    if (match := re.fullmatch(r'quarantine_day(\d+)', column))
 )
s3_daily_frames = []
for day in day_numbers:
    day_data = s3_users[['id', 'group', *s3_items]].copy()
    day_data['q'] = s3_users[f'quarantine_day{day}']
    day_data['nq'] = s3_users[f'no_quarantine_day{day}']
    day_data['day'] = day
    s3_daily_frames.append(day_data)

s3_daily = pd.concat(s3_daily_frames, ignore_index=True)
s3_daily = s3_daily[(s3_daily['q'] + s3_daily['nq']) > 0].copy()


def slope_contrast(model, terms):
    contrast = np.zeros(len(model.params))
    for term in terms:
        contrast[model.params.index.get_loc(term)] = 1
    estimate = float(contrast @ model.params)
    variance = float(contrast @ model.cov_params() @ contrast)
    standard_error = np.sqrt(variance)
    statistic = estimate / standard_error
    p_value = 2 * stats.norm.sf(abs(statistic))
    return {
        'log_or': estimate,
        'or': np.exp(estimate),
        'ci_low': np.exp(estimate - 1.96 * standard_error),
        'ci_high': np.exp(estimate + 1.96 * standard_error),
        'p': p_value,
    }


s3_results = []
for item, label in s3_items.items():
    participant_data = s3_users.dropna(subset=[item]).copy()
    item_mean = participant_data[item].mean()
    participant_data['item_centered'] = participant_data[item] - item_mean
    daily_data = s3_daily.dropna(subset=[item]).copy()
    daily_data['item_centered'] = daily_data[item] - item_mean

    spearman = stats.spearmanr(
        participant_data[item],
        participant_data['quarantine_rate'],
    )

    gee_main = sm.GEE.from_formula(
        'q ~ item_centered + C(group)',
        groups='id',
        data=daily_data,
        family=sm.families.Binomial(),
        cov_struct=sm.cov_struct.Exchangeable(),
    ).fit()
    gee_interaction = sm.GEE.from_formula(
        'q ~ item_centered * C(group)',
        groups='id',
        data=daily_data,
        family=sm.families.Binomial(),
        cov_struct=sm.cov_struct.Exchangeable(),
    ).fit()

    qb_main = smf.glm(
        'quarantine_rate ~ item_centered + C(group)',
        data=participant_data,
        family=sm.families.Binomial(),
        var_weights=participant_data['total_trials'],
    ).fit(scale='X2')
    qb_interaction = smf.glm(
        'quarantine_rate ~ item_centered * C(group)',
        data=participant_data,
        family=sm.families.Binomial(),
        var_weights=participant_data['total_trials'],
    ).fit(scale='X2')

    interaction_term = 'item_centered:C(group)[T.2]'
    gee_main_slope = slope_contrast(gee_main, ['item_centered'])
    gee_g1_slope = slope_contrast(gee_interaction, ['item_centered'])
    gee_g2_slope = slope_contrast(
        gee_interaction,
        ['item_centered', interaction_term],
    )
    gee_difference = slope_contrast(gee_interaction, [interaction_term])
    qb_main_slope = slope_contrast(qb_main, ['item_centered'])
    qb_g1_slope = slope_contrast(qb_interaction, ['item_centered'])
    qb_g2_slope = slope_contrast(
        qb_interaction,
        ['item_centered', interaction_term],
    )
    qb_difference = slope_contrast(qb_interaction, [interaction_term])

    s3_results.append({
        'item': item,
        'construct': label,
        'participants': len(participant_data),
        'participant_days': len(daily_data),
        'spearman_rho': spearman.statistic,
        'spearman_p_raw': spearman.pvalue,
        'gee_main_or': gee_main_slope['or'],
        'gee_main_ci_low': gee_main_slope['ci_low'],
        'gee_main_ci_high': gee_main_slope['ci_high'],
        'gee_main_p_raw': gee_main_slope['p'],
        'gee_g1_or': gee_g1_slope['or'],
        'gee_g1_ci_low': gee_g1_slope['ci_low'],
        'gee_g1_ci_high': gee_g1_slope['ci_high'],
        'gee_g1_p': gee_g1_slope['p'],
        'gee_g2_or': gee_g2_slope['or'],
        'gee_g2_ci_low': gee_g2_slope['ci_low'],
        'gee_g2_ci_high': gee_g2_slope['ci_high'],
        'gee_g2_p': gee_g2_slope['p'],
        'gee_interaction_or': gee_difference['or'],
        'gee_interaction_ci_low': gee_difference['ci_low'],
        'gee_interaction_ci_high': gee_difference['ci_high'],
        'gee_interaction_p_raw': gee_difference['p'],
        'qb_main_or': qb_main_slope['or'],
        'qb_main_ci_low': qb_main_slope['ci_low'],
        'qb_main_ci_high': qb_main_slope['ci_high'],
        'qb_main_p_raw': qb_main_slope['p'],
        'qb_g1_or': qb_g1_slope['or'],
        'qb_g1_ci_low': qb_g1_slope['ci_low'],
        'qb_g1_ci_high': qb_g1_slope['ci_high'],
        'qb_g1_p': qb_g1_slope['p'],
        'qb_g2_or': qb_g2_slope['or'],
        'qb_g2_ci_low': qb_g2_slope['ci_low'],
        'qb_g2_ci_high': qb_g2_slope['ci_high'],
        'qb_g2_p': qb_g2_slope['p'],
        'qb_interaction_or': qb_difference['or'],
        'qb_interaction_ci_low': qb_difference['ci_low'],
        'qb_interaction_ci_high': qb_difference['ci_high'],
        'qb_interaction_p_raw': qb_difference['p'],
        'qb_pearson_scale': qb_interaction.scale,
    })

s3_results = pd.DataFrame(s3_results)
for raw_column, adjusted_column in [
    ('spearman_p_raw', 'spearman_p_holm'),
    ('gee_main_p_raw', 'gee_main_p_holm'),
    ('qb_main_p_raw', 'qb_main_p_holm'),
    ('gee_interaction_p_raw', 'gee_interaction_p_holm'),
    ('qb_interaction_p_raw', 'qb_interaction_p_holm'),
]:
    s3_results[adjusted_column] = multipletests(
        s3_results[raw_column],
        method='holm',
    )[1]

s3_output = Path('output/202/S3-quarantine-exploratory-results.csv')
s3_output.parent.mkdir(parents=True, exist_ok=True)
s3_results.to_csv(s3_output, index=False)
display(s3_results.round(4))
print(f'Saved: {s3_output}')

,item,construct,participants,participant_days,spearman_rho,spearman_p_raw,gee_main_or,gee_main_ci_low,gee_main_ci_high,gee_main_p_raw,...,qb_interaction_or,qb_interaction_ci_low,qb_interaction_ci_high,qb_interaction_p_raw,qb_pearson_scale,spearman_p_holm,gee_main_p_holm,qb_main_p_holm,gee_interaction_p_holm,qb_interaction_p_holm
0,S3_Q1,Perceived realism,48,402,0.1526,0.3004,1.1980,0.9276,1.5471,0.1662,...,1.1220,0.4374,2.8779,0.8107,1.0353,0.9011,0.665,1.0,1.0000,1.0
1,S3_Q2,Concern about virtual infection,48,402,0.0194,0.8959,0.9200,0.7118,1.1890,0.5239,...,0.8860,0.4602,1.7057,0.7173,1.0119,1.0000,1.000,1.0,1.0000,1.0
2,S3_Q3,Real-life correspondence of quarantine decisions,48,402,0.0119,0.9361,0.9344,0.6828,1.2788,0.6719,...,1.6386,0.6257,4.2911,0.3147,1.1019,1.0000,1.000,1.0,0.4817,1.0
3,S3_Q4,Difficulty understanding game information,48,402,0.2241,0.1258,1.2187,0.7971,1.8632,0.3612,...,0.6312,0.2749,1.4493,0.2779,0.9192,0.5031,1.000,1.0,0.8918,1.0


Saved: output/202/S3-quarantine-exploratory-results.csv
